# HDBSCAN coverage test — graph email features per MISP file

For each MISP JSON, we take **canonical `external_id`s** (same rule as graph build / `analysis/scripts/filter_misp_exclude_ground_truth.py`: `Event.external_id` stripped, else `str(email_index)` with `email_index` defaulting to the list index).

We load **`data["email"].x`** slices via `load_graph_email_features_for_external_ids` + the **matching** `*_hetero.meta.json` — alignment is **by ID string**, not by JSON row ↔ graph row.

**Pairing:** each incidents file should use the hetero checkpoint that was built from that universe (defaults below: `large` → `incidents-lake-misp-large_hetero.pt`, lake → `incidents-lake-misp_hetero.pt`). Adjust `CONFIGS` if your artifacts live elsewhere.

**Coverage reported:** among emails **present in the graph**, fraction HDBSCAN labels as **noise (`-1`)** vs **assigned to a cluster**.

`FEATURE_MODE`: `auto` matches experiment 3 (projected graphs → 128-d SBERT block). For a **raw assembler** checkpoint with a large `email.x` width, set `FEATURE_MODE="raw_subject_body"` explicitly.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

_here = Path.cwd().resolve()
PROJECT_ROOT = _here if (_here / "pipeline_config.json").is_file() else _here.parent
assert (PROJECT_ROOT / "pipeline_config.json").is_file(), f"Set cwd to repo root or analysis/; got {PROJECT_ROOT}"
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from analysis.utils.email_teacher_contrastive_eval import matrix_clustering_sanity
from analysis.utils.email_teacher_contrastive_features import load_graph_email_features_for_external_ids

# Pair each MISP JSON with the hetero graph built from that email universe.
CONFIGS = [
    {
        "name": "incidents-lake-misp-large",
        "misp_json": PROJECT_ROOT / "data" / "misp" / "incidents-lake-misp-large.json",
        "graph_pt": PROJECT_ROOT / "core" / "graph" / "output" / "incidents-lake-misp-large_hetero.pt",
    },
    {
        "name": "incidents-lake-misp",
        "misp_json": PROJECT_ROOT / "data" / "misp" / "incidents-lake-misp.json",
        "graph_pt": PROJECT_ROOT / "core" / "graph" / "output" / "incidents-lake-misp_hetero.pt",
    },
]

# auto → projected_bert128 on typical 160-d checkpoints; see module docstring for raw layouts.
FEATURE_MODE = "auto"

MIN_CLUSTER_SIZE = 10
MIN_SAMPLES: int | None = None  # None → HDBSCAN default linkage to min_cluster_size
HDBSCAN_METRIC = "cosine"
HDBSCAN_CLUSTER_SELECTION_METHOD = "leaf"
CLUSTER_SELECTION_EPSILON = 0.0
STANDARDIZE_COLUMNS = False

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

In [2]:
def _to_str(val):
    if val is None:
        return ""
    return val if isinstance(val, str) else str(val)


def misp_event_canonical_id(ev, list_index: int) -> str:
    """Align with graph MISP parsing / filter_misp_exclude_ground_truth."""
    if not isinstance(ev, dict):
        return str(list_index)
    event = ev.get("Event")
    if not isinstance(event, dict):
        return str(list_index)
    email_index = event.get("email_index", list_index)
    ext = _to_str(event.get("external_id", "")).strip()
    return ext or str(email_index)


def external_ids_from_misp_path(path: Path, *, dedupe: bool = True):
    path = Path(path)
    with open(path, "r", encoding="utf-8-sig") as f:
        events = json.load(f)
    if not isinstance(events, list):
        raise TypeError(f"Expected JSON array at {path}; got {type(events).__name__}")
    raw_ids = [misp_event_canonical_id(ev, i) for i, ev in enumerate(events)]
    dup_entries = len(raw_ids) - len(set(raw_ids))
    stats = {
        "n_events": len(events),
        "n_ids_in_list_order": len(raw_ids),
        "n_unique_external_ids": len(set(raw_ids)),
        "n_duplicate_list_entries": dup_entries,
    }
    if not dedupe:
        return raw_ids, stats
    seen: set[str] = set()
    ordered: list[str] = []
    for eid in raw_ids:
        if eid in seen:
            continue
        seen.add(eid)
        ordered.append(eid)
    return ordered, stats


def run_hdbscan_noise_stats(X: np.ndarray, *, min_cluster_size: int, min_samples: int | None, metric: str, cluster_selection_method: str, cluster_selection_epsilon: float, standardize_columns: bool):
    import hdbscan
    from sklearn.preprocessing import StandardScaler

    X = np.asarray(X, dtype=np.float64, order="C")
    if X.shape[0] < 2:
        return None
    if standardize_columns:
        X = StandardScaler().fit_transform(X)
    metric_s = str(metric).strip().lower()
    want_cosine = metric_s in ("cosine", "cosine_distance")
    if want_cosine:
        norms = np.linalg.norm(X, axis=1, keepdims=True)
        np.maximum(norms, 1e-12, out=norms)
        X = X / norms
        metric_for_hdbscan = "euclidean"
    else:
        metric_for_hdbscan = metric_s
    mcs = int(min_cluster_size)
    ms = mcs if min_samples is None else int(min_samples)
    csm = str(cluster_selection_method).strip().lower()
    kw = dict(
        min_cluster_size=mcs,
        min_samples=ms,
        metric=metric_for_hdbscan,
        cluster_selection_epsilon=float(cluster_selection_epsilon),
        allow_single_cluster=False,
    )
    try:
        clusterer = hdbscan.HDBSCAN(**kw, cluster_selection_method=csm)
    except TypeError:
        clusterer = hdbscan.HDBSCAN(**kw)
    labels = np.asarray(clusterer.fit_predict(X), dtype=np.int64)
    n = int(labels.shape[0])
    n_noise = int((labels == -1).sum())
    n_non_noise = n - n_noise
    uniq = {int(x) for x in labels.tolist()}
    if -1 in uniq:
        uniq.remove(-1)
    return {
        "n_points": n,
        "n_noise": n_noise,
        "n_non_noise": n_non_noise,
        "frac_noise": n_noise / n,
        "frac_non_noise": n_non_noise / n,
        "n_clusters_pred": len(uniq),
        "labels": labels,
    }

In [3]:
rows: list[dict] = []
details: list[tuple[str, list[str], dict | None]] = []

for cfg in CONFIGS:
    name = cfg["name"]
    mj = Path(cfg["misp_json"])
    gp = Path(cfg["graph_pt"])
    meta_json = gp.with_suffix(".meta.json")

    if not mj.is_file():
        print(f"SKIP {name}: missing MISP JSON\n  {mj}")
        continue
    if not gp.is_file():
        print(f"SKIP {name}: missing graph .pt\n  {gp}")
        continue
    if not meta_json.is_file():
        print(f"SKIP {name}: missing meta\n  {meta_json}")
        continue

    eids, id_stats = external_ids_from_misp_path(mj, dedupe=True)
    X, mask, finfo = load_graph_email_features_for_external_ids(
        gp,
        meta_json,
        eids,
        feature_mode=FEATURE_MODE,
        to_undirected=True,
    )

    X_fit = np.asarray(X[mask], dtype=np.float64)
    present_ix = np.flatnonzero(mask)
    eids_fit = [eids[i] for i in present_ix]

    sanity = matrix_clustering_sanity(X_fit, tag=name)
    res = run_hdbscan_noise_stats(
        X_fit,
        min_cluster_size=MIN_CLUSTER_SIZE,
        min_samples=MIN_SAMPLES,
        metric=HDBSCAN_METRIC,
        cluster_selection_method=HDBSCAN_CLUSTER_SELECTION_METHOD,
        cluster_selection_epsilon=CLUSTER_SELECTION_EPSILON,
        standardize_columns=STANDARDIZE_COLUMNS,
    )

    row = {
        "dataset": name,
        "n_misp_events": id_stats["n_events"],
        "n_unique_external_ids_misp": id_stats["n_unique_external_ids"],
        "n_duplicate_misp_list_entries": id_stats["n_duplicate_list_entries"],
        "feature_mode_resolved": finfo["feature_mode_resolved"],
        "feature_dim": finfo["output_dim"],
        "n_requested_in_graph_order": finfo["n_requested"],
        "n_present_in_graph": finfo["n_present"],
        "n_missing_in_graph": finfo["n_missing"],
        "frac_misp_ids_missing_from_graph": finfo["n_missing"] / max(1, finfo["n_requested"]),
        "matrix_all_finite": sanity["all_finite"],
        "n_near_zero_rows": sanity["n_near_zero_rows"],
    }
    if res is not None:
        row["hdbscan_n_points"] = res["n_points"]
        row["n_noise"] = res["n_noise"]
        row["n_non_noise"] = res["n_non_noise"]
        row["frac_noise"] = res["frac_noise"]
        row["frac_non_noise"] = res["frac_non_noise"]
        row["n_clusters_ex_noise"] = res["n_clusters_pred"]
    else:
        row["hdbscan_note"] = "skipped (<2 points in graph)"

    rows.append(row)
    details.append((name, eids_fit, res))
    print(f"--- {name} ---")
    print("misp_json:", mj)
    print("graph_pt:", gp)
    print("feature load:", {k: finfo[k] for k in ("feature_mode_resolved", "slice", "output_dim", "n_present", "n_missing")})

summary = pd.DataFrame(rows)
summary

c:\Users\aar\Desktop\GNN-Campaign-Detection\.venv311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


--- incidents-lake-misp-large ---
misp_json: C:\Users\aar\Desktop\GNN-Campaign-Detection\data\misp\incidents-lake-misp-large.json
graph_pt: C:\Users\aar\Desktop\GNN-Campaign-Detection\core\graph\output\incidents-lake-misp-large_hetero.pt
feature load: {'feature_mode_resolved': 'projected_bert128', 'slice': [0, 128], 'output_dim': 128, 'n_present': 7333, 'n_missing': 0}
--- incidents-lake-misp ---
misp_json: C:\Users\aar\Desktop\GNN-Campaign-Detection\data\misp\incidents-lake-misp.json
graph_pt: C:\Users\aar\Desktop\GNN-Campaign-Detection\core\graph\output\incidents-lake-misp_hetero.pt
feature load: {'feature_mode_resolved': 'projected_bert128', 'slice': [0, 128], 'output_dim': 128, 'n_present': 4437, 'n_missing': 0}


,dataset,n_misp_events,n_unique_external_ids_misp,n_duplicate_misp_list_entries,feature_mode_resolved,feature_dim,n_requested_in_graph_order,n_present_in_graph,n_missing_in_graph,frac_misp_ids_missing_from_graph,matrix_all_finite,n_near_zero_rows,hdbscan_n_points,n_noise,n_non_noise,frac_noise,frac_non_noise,n_clusters_ex_noise
0,incidents-lake-misp-large,7333,7333,0,projected_bert128,128,7333,7333,0,0.0,True,0,7333,4607,2726,0.628256,0.371744,135
1,incidents-lake-misp,4437,4437,0,projected_bert128,128,4437,4437,0,0.0,True,0,4437,2731,1706,0.615506,0.384494,83


In [4]:
# Label distribution (cluster sizes + noise) per dataset
for name, eids_fit, res in details:
    print(f"\n=== {name} ===")
    if res is None:
        print("No HDBSCAN result.")
        continue
    lab = res["labels"]
    print(f"points={len(eids_fit)}  noise={res['n_noise']} ({res['frac_noise']:.4f})  clustered={res['n_non_noise']} ({res['frac_non_noise']:.4f})")
    print(f"predicted clusters (ex noise): {res['n_clusters_pred']}")
    in_cluster = lab >= 0
    if in_cluster.any():
        _, counts = np.unique(lab[in_cluster], return_counts=True)
        counts_sorted = np.sort(counts)[::-1]
        print(f"cluster size stats: min={counts_sorted.min()} max={counts_sorted.max()} median={np.median(counts_sorted):.0f}")
        print(f"top 8 cluster sizes: {counts_sorted[:8].tolist()}")


=== incidents-lake-misp-large ===
points=7333  noise=4607 (0.6283)  clustered=2726 (0.3717)
predicted clusters (ex noise): 135
cluster size stats: min=11 max=73 median=17
top 8 cluster sizes: [73, 71, 54, 52, 51, 49, 48, 39]

=== incidents-lake-misp ===
points=4437  noise=2731 (0.6155)  clustered=1706 (0.3845)
predicted clusters (ex noise): 83
cluster size stats: min=10 max=73 median=17
top 8 cluster sizes: [73, 71, 49, 43, 40, 38, 36, 36]
